Imagine you're a bank or a payment company (like Vesta, the real company behind this dataset). Every time someone swipes a card online, you have milliseconds to decide: approve or block?

Two ways to get this wrong, and both cost real money:

Miss a fraud (approve it) → you eat the loss, sometimes thousands of dollars
Flag a real customer as fraud (block it) → customer gets embarrassed/angry, may leave the bank, and you lose a legitimate sale

So the real challenge isn't "detect fraud" — it's catching fraud while barely bothering real customers, because those two goals fight each other.

In [1]:
# 1. IMPORTS
import pandas as pd
import numpy as np

# 2. LOAD DATA
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')

print("Transaction shape:", train_transaction.shape)
print("Identity shape:", train_identity.shape)

Transaction shape: (590540, 394)
Identity shape: (144233, 41)


In [2]:
# 3. INSPECT TARGET VARIABLE
fraud_counts = train_transaction['isFraud'].value_counts()
fraud_pct = train_transaction['isFraud'].value_counts(normalize=True) * 100

print(fraud_counts)
print(fraud_pct)

# 4. CHECK IDENTITY OVERLAP
matched = train_transaction['TransactionID'].isin(train_identity['TransactionID']).sum()
print(f"\nTransactions WITH identity info: {matched} ({matched/len(train_transaction)*100:.1f}%)")
print(f"Transactions WITHOUT identity info: {len(train_transaction)-matched} ({100-matched/len(train_transaction)*100:.1f}%)")

isFraud
0    569877
1     20663
Name: count, dtype: int64
isFraud
0    96.500999
1     3.499001
Name: proportion, dtype: float64

Transactions WITH identity info: 144233 (24.4%)
Transactions WITHOUT identity info: 446307 (75.6%)


In [3]:
# 5. MERGE TRANSACTION + IDENTITY (LEFT JOIN — keep all transactions)
df = train_transaction.merge(train_identity, on='TransactionID', how='left')

# 6. ADD THE "HAS IDENTITY" FLAG — captures the missingness pattern itself
df['has_identity'] = df['TransactionID'].isin(train_identity['TransactionID']).astype(int)

print("Merged shape:", df.shape)
print(df['has_identity'].value_counts())

Merged shape: (590540, 435)
has_identity
0    446307
1    144233
Name: count, dtype: int64


In [4]:
# 7. CHECK FRAUD RATE BY HAS_IDENTITY
fraud_by_identity = df.groupby('has_identity')['isFraud'].mean() * 100
print(fraud_by_identity)

has_identity
0    2.093850
1    7.847025
Name: isFraud, dtype: float64


In [5]:
# 8. CHECK MISSING VALUES ACROSS ALL COLUMNS
missing_summary = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing_summary / len(df)) * 100

print(missing_pct.head(20))
print(f"\nColumns missing >90% of data: {(missing_pct > 90).sum()}")
print(f"Columns missing >50% of data: {(missing_pct > 50).sum()}")

id_24    99.196159
id_25    99.130965
id_07    99.127070
id_08    99.127070
id_21    99.126393
id_26    99.125715
id_27    99.124699
id_23    99.124699
id_22    99.124699
dist2    93.628374
D7       93.409930
id_18    92.360721
D13      89.509263
D14      89.469469
D12      89.041047
id_04    88.768923
id_03    88.768923
D6       87.606767
id_33    87.589494
id_09    87.312290
dtype: float64

Columns missing >90% of data: 12
Columns missing >50% of data: 214


In [6]:
# 9. INVESTIGATE WHAT id_ AND D COLUMNS ACTUALLY REPRESENT

# 9a. Compare missingness pattern across id_ columns
id_cols = [c for c in df.columns if c.startswith('id_')]
print("id_ columns missing %:")
print(df[id_cols].isnull().mean().sort_values(ascending=False) * 100)

# 9b. Does id_ missingness align exactly with has_identity?
print("\nRows where has_identity=0 but id_01 is NOT null:", 
      ((df['has_identity'] == 0) & (df['id_01'].notnull())).sum())
print("Rows where has_identity=1 but id_01 IS null:", 
      ((df['has_identity'] == 1) & (df['id_01'].isnull())).sum())

# 9c. Inspect D-column raw values and ranges
d_cols = [c for c in df.columns if c.startswith('D') and c[1:].isdigit()]
print("\nD columns summary:")
print(df[d_cols].describe().T[['min', 'max', 'mean']])

# 9d. Check correlation between D columns and TransactionDT (time elapsed since start of dataset)
print("\nCorrelation of D columns with TransactionDT:")
print(df[d_cols + ['TransactionDT']].corr()['TransactionDT'])

id_ columns missing %:
id_24    99.196159
id_25    99.130965
id_08    99.127070
id_07    99.127070
id_21    99.126393
id_26    99.125715
id_27    99.124699
id_22    99.124699
id_23    99.124699
id_18    92.360721
id_03    88.768923
id_04    88.768923
id_33    87.589494
id_09    87.312290
id_10    87.312290
id_30    86.865411
id_32    86.861855
id_34    86.824771
id_14    86.445626
id_13    78.440072
id_16    78.098012
id_05    76.823755
id_06    76.823755
id_20    76.418024
id_19    76.408372
id_17    76.399736
id_31    76.245132
id_02    76.145223
id_29    76.127273
id_11    76.127273
id_28    76.127273
id_37    76.126088
id_35    76.126088
id_15    76.126088
id_38    76.126088
id_36    76.126088
id_01    75.576083
id_12    75.576083
dtype: float64

Rows where has_identity=0 but id_01 is NOT null: 0
Rows where has_identity=1 but id_01 IS null: 0

D columns summary:
       min          max        mean
D1     0.0   640.000000   94.347568
D2     0.0   640.000000  169.563231
D3     0.0   

In [8]:
# 10. CHECK FOR DUPLICATE ROWS
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows: {duplicate_count}")

# 11. CHECK FOR DUPLICATE TRANSACTION IDs (should be zero — it's our unique key)
duplicate_ids = df['TransactionID'].duplicated().sum()
print(f"Duplicate TransactionIDs: {duplicate_ids}")

Duplicate rows: 0
Duplicate TransactionIDs: 0


In [9]:
# 12. CHECK DATA TYPES ACROSS THE DATASET
print(df.dtypes.value_counts())

# 13. LOOK AT THE ACTUAL CATEGORICAL COLUMNS (Vesta documented these as categorical)
categorical_cols = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 
                     'M1','M2','M3','M4','M5','M6','M7','M8','M9']
for col in categorical_cols:
    print(f"\n{col}: {df[col].nunique()} unique values")
    print(df[col].value_counts(dropna=False).head(5))

float64    399
object      31
int64        5
Name: count, dtype: int64

ProductCD: 5 unique values
ProductCD
W    439670
C     68519
R     37699
H     33024
S     11628
Name: count, dtype: int64

card4: 4 unique values
card4
visa                384767
mastercard          189217
american express      8328
discover              6651
NaN                   1577
Name: count, dtype: int64

card6: 4 unique values
card6
debit              439938
credit             148986
NaN                  1571
debit or credit        30
charge card            15
Name: count, dtype: int64

P_emaildomain: 59 unique values
P_emaildomain
gmail.com        228355
yahoo.com        100934
NaN               94456
hotmail.com       45250
anonymous.com     36998
Name: count, dtype: int64

R_emaildomain: 60 unique values
R_emaildomain
NaN              453249
gmail.com         57147
hotmail.com       27509
anonymous.com     20529
yahoo.com         11842
Name: count, dtype: int64

M1: 2 unique values
M1
T      319415
NaN 

In [10]:
# 14. FIND THE REMAINING OBJECT-TYPE COLUMNS WE HAVEN'T INSPECTED YET
object_cols = df.select_dtypes(include='object').columns.tolist()
already_checked = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain',
                    'M1','M2','M3','M4','M5','M6','M7','M8','M9']
remaining = [c for c in object_cols if c not in already_checked]
print(f"Remaining object columns ({len(remaining)}):", remaining)

Remaining object columns (17): ['id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


In [11]:
# 15. CHECK IF M1 MISSINGNESS CLUSTERS AROUND A SPECIFIC ProductCD
print(df.groupby('ProductCD')['M1'].apply(lambda x: x.isnull().mean() * 100))

# 16. DO THE SAME FOR A FEW OTHER M-COLUMNS TO SEE IF THE PATTERN HOLDS
for col in ['M2', 'M4', 'M7']:
    print(f"\n{col} missing % by ProductCD:")
    print(df.groupby('ProductCD')[col].apply(lambda x: x.isnull().mean() * 100))

ProductCD
C    100.000000
H    100.000000
R    100.000000
S    100.000000
W     27.345509
Name: M1, dtype: float64

M2 missing % by ProductCD:
ProductCD
C    100.000000
H    100.000000
R    100.000000
S    100.000000
W     27.345509
Name: M2, dtype: float64

M4 missing % by ProductCD:
ProductCD
C      2.530685
H    100.000000
R    100.000000
S    100.000000
W     44.887984
Name: M4, dtype: float64

M7 missing % by ProductCD:
ProductCD
C    100.000000
H    100.000000
R    100.000000
S    100.000000
W     44.441286
Name: M7, dtype: float64


In [12]:
# 17. INSPECT card6 CATEGORIES IN DETAIL
print(df['card6'].value_counts(dropna=False))

# Check if "debit or credit" and "charge card" rows show anything unusual (e.g. all from one ProductCD, or all fraud)
print("\ncard6 == 'debit or credit' rows - ProductCD breakdown:")
print(df[df['card6'] == 'debit or credit']['ProductCD'].value_counts())

print("\ncard6 == 'debit or credit' fraud rate:", df[df['card6'] == 'debit or credit']['isFraud'].mean())
print("card6 == 'charge card' fraud rate:", df[df['card6'] == 'charge card']['isFraud'].mean())
print("Overall fraud rate:", df['isFraud'].mean())

card6
debit              439938
credit             148986
NaN                  1571
debit or credit        30
charge card            15
Name: count, dtype: int64

card6 == 'debit or credit' rows - ProductCD breakdown:
ProductCD
W    30
Name: count, dtype: int64

card6 == 'debit or credit' fraud rate: 0.0
card6 == 'charge card' fraud rate: 0.0
Overall fraud rate: 0.03499000914417313


In [13]:
# 18. INSPECT DeviceType AND DeviceInfo
print("DeviceType:")
print(df['DeviceType'].value_counts(dropna=False))

print("\nDeviceInfo (top 15):")
print(df['DeviceInfo'].value_counts(dropna=False).head(15))
print(f"\nDeviceInfo unique values: {df['DeviceInfo'].nunique()}")

DeviceType:
DeviceType
NaN        449730
desktop     85165
mobile      55645
Name: count, dtype: int64

DeviceInfo (top 15):
DeviceInfo
NaN                            471874
Windows                         47722
iOS Device                      19782
MacOS                           12573
Trident/7.0                      7440
rv:11.0                          1901
rv:57.0                           962
SM-J700M Build/MMB29K             549
SM-G610M Build/MMB29K             461
SM-G531H Build/LMY48B             410
rv:59.0                           362
SM-G935F Build/NRD90M             334
SM-G955U Build/NRD90M             328
SM-G532M Build/MMB29T             316
ALE-L23 Build/HuaweiALE-L23       312
Name: count, dtype: int64

DeviceInfo unique values: 1786


In [14]:
# 19. DOES DeviceType MISSINGNESS MATCH has_identity?
print("Rows where has_identity=0 but DeviceType is NOT null:",
      ((df['has_identity'] == 0) & (df['DeviceType'].notnull())).sum())
print("Rows where has_identity=1 but DeviceType IS null:",
      ((df['has_identity'] == 1) & (df['DeviceType'].isnull())).sum())

Rows where has_identity=0 but DeviceType is NOT null: 0
Rows where has_identity=1 but DeviceType IS null: 3423


In [15]:
# SAVE MERGED + INVESTIGATED DATAFRAME TO PROCESSED FOLDER
df.to_csv('/kaggle/working/train_merged.csv', index=False)
print("Saved:", df.shape)

Saved: (590540, 435)
